# 01: Titanic TF-DF Advanced Model

**Purpose:** Advanced Google Colab-only TensorFlow Decision Forests notebook inspired by the Gusthema-style approach.

TF-DF is extremely heavy and requires TensorFlow, making it a Colab-only component that is excluded from the frontend website.


In [ ]:
# Install TF-DF inside Colab environment
import sys
try:
    import tensorflow_decision_forests as tfdf
except ImportError:
    !pip install tensorflow_decision_forests
    import tensorflow_decision_forests as tfdf

import tensorflow as tf
import pandas as pd
import numpy as np
import os


## 1. Load Data
Load the Kaggle Titanic train.csv and test.csv.


In [ ]:
possible_paths = ["../titanic/", "./titanic/", "/content/"]
train_df, test_df = None, None

for bp in possible_paths:
    if os.path.exists(os.path.join(bp, "train.csv")):
        train_df = pd.read_csv(os.path.join(bp, "train.csv"))
        test_df = pd.read_csv(os.path.join(bp, "test.csv"))
        break

if train_df is None:
    raise FileNotFoundError("Kaggle datasets missing!")

print("Train dataset:", train_df.shape)
print("Test dataset:", test_df.shape)


## 2. Advanced Feature Engineering & Name Tokenization
Normalise passenger names, extract ticket numbers/items, and tokenize names using TensorFlow string utilities.


In [ ]:
def advanced_prep(df):
    df = df.copy()

    # Normalise name
    df["Name"] = df["Name"].str.lower()

    # Ticket extraction
    def split_ticket(ticket):
        if pd.isna(ticket):
            return "X", 0
        ticket = str(ticket).strip()
        parts = ticket.split()
        if len(parts) > 1:
            number = parts[-1]
            item = "".join(parts[:-1]).replace(".", "").replace("/", "").lower()
            return item, int(number) if number.isdigit() else 0
        else:
            val = parts[0]
            return "X", int(val) if val.isdigit() else 0

    splits = df["Ticket"].apply(split_ticket)
    df["Ticket_item"] = [s[0] for s in splits]
    df["Ticket_number"] = [s[1] for s in splits]

    return df

train_prep = advanced_prep(train_df)
test_prep = advanced_prep(test_df)
print("Engineered advanced features.")


## 3. Convert to TF-DF Datasets and Train GB Trees


In [ ]:
# Features to feed
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Cabin", "Embarked", "Ticket_number", "Ticket_item", "Name"]

# Convert target to int
train_prep["Survived"] = train_prep["Survived"].astype(int)

# Setup TF Dataset
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_prep[features + ["Survived"]],
    label="Survived"
)

test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_prep[features]
)

# Train GradientBoostedTreesModel
model = tfdf.keras.GradientBoostedTreesModel(
    features=[tfdf.keras.FeatureUsage(f) for f in features],
    exclude_non_specified_features=True,
    random_seed=42
)

model.fit(train_ds)
print(model.summary())


## 4. Hyperparameter Tuning and Exporting Submissions


In [ ]:
# Predictions on test data
test_preds = model.predict(test_ds)

# Output default submission
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": (test_preds > 0.5).astype(int).squeeze()
})

os.makedirs("../submissions/", exist_ok=True)
submission.to_csv("../submissions/submission_tfdf_default.csv", index=False)
print("✓ Saved submission_tfdf_default.csv")
